# Model Prediction

## Import necessary libraries

In [1]:
%pip install -qq -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Add current directory to Python path for imports
import os
import sys

# Add the parent directory (project root) to Python path so we can import from src
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
# Utility Functions
from src.utils import create_spark_session

# Create Spark session
spark, sedona = create_spark_session(app_name="ModelPredictionSpark")

## Loading Datasets

In [4]:
from src.utils import read_config_path

# Load data using configuration file
filepath = read_config_path(key="raw_data_path")

df = spark.read.csv(
    filepath,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"',
)

df.show(10)

+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|  ticket_id|               type|        organization|             comment|               photo|         photo_after|            coords|             address|subdistrict|district|     province|           timestamp|    state|star|count_reopen|       last_activity|
+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|2021-FYJTFP|        {ความสะอาด}|          เขตบางซื่อ|             ขยะเยอะ|https://storage.g...|                NULL|100.53084,13.81865|12/14 ถนน กรุงเทพ...|       NULL|    NULL|กรุงเทพมหานคร|2021-09-03 19:51:..

## Sample data

In [ ]:
sample_df = df.limit(100)
sample_df.show(10)

+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|  ticket_id|               type|        organization|             comment|               photo|         photo_after|            coords|             address|subdistrict|district|     province|           timestamp|    state|star|count_reopen|       last_activity|
+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|2021-FYJTFP|        {ความสะอาด}|          เขตบางซื่อ|             ขยะเยอะ|https://storage.g...|                NULL|100.53084,13.81865|12/14 ถนน กรุงเทพ...|       NULL|    NULL|กรุงเทพมหานคร|2021-09-03 19:51:..

---

## Applying Cleansing Pipeline

In [6]:
from src.pipelines_spark import CleansingPipelineSpark

cleansing_pipeline = CleansingPipelineSpark(spark, sedona)
df_cleansed = cleansing_pipeline.transform(sample_df)

df_cleansed.show(10)

+-----------+-------------------+--------------------+--------------------+--------------------+-----------+--------+-------------+--------------+---------------+--------------+------------------+-------------------+------------------+---------------+---------+--------+------+
|  ticket_id|               type|        organization|             comment|             address|subdistrict|district|     province|timestamp_date|timestamp_month|timestamp_year|last_activity_date|last_activity_month|last_activity_year|resolution_time|longitude|latitude|status|
+-----------+-------------------+--------------------+--------------------+--------------------+-----------+--------+-------------+--------------+---------------+--------------+------------------+-------------------+------------------+---------------+---------+--------+------+
|2021-CGPMUN|{น้ำท่วม,ร้องเรียน}|เขตประเวศ,ฝ่ายโยธ...|น้ำท่วมเวลาฝนตกแล...|189 เฉลิมพระเกียร...|    หนองบอน|  ประเวศ|กรุงเทพมหานคร|            19|              9|    

In [7]:
df_cleansed.printSchema()

root
 |-- ticket_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- organization: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- address: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- timestamp_date: integer (nullable = true)
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- last_activity_date: integer (nullable = true)
 |-- last_activity_month: integer (nullable = true)
 |-- last_activity_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- status: string (nullable = true)



---

## Applying Model Preparation Pipeline

In [8]:
from src.pipelines_spark import ModelPrepPipelineSpark

preparing_pipeline = ModelPrepPipelineSpark()
df_prepared = preparing_pipeline.transform(df_cleansed)

df_prepared.show(10)

+---------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+
|timestamp_month|timestamp_year|resolution_time|     address_encoded|     latlong_encoded|organization_encoded|        type_encoded|
+---------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+
|              9|          2021|            275|(2048,[834,1804],...|[13.67891,100.66709]|(1786,[10,53],[1....|(25,[7,8],[1.0,1.0])|
|              9|          2021|            253|(2048,[348,426],[...| [13.7206,100.52649]|   (1786,[49],[1.0])|     (25,[14],[1.0])|
|             12|          2021|            246|(2048,[802,1656],...| [13.8228,100.59165]|(1786,[31,108],[1...|(25,[0,8],[1.0,1.0])|
|             12|          2021|            456|(2048,[802,1656],...| [13.8091,100.59131]|(1786,[31,172],[1...|      (25,[1],[1.0])|
|             12|          2021|            516|(2048,[1025,1114]...|

In [9]:
df_prepared.printSchema()

root
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- address_encoded: vector (nullable = true)
 |-- latlong_encoded: vector (nullable = true)
 |-- organization_encoded: vector (nullable = true)
 |-- type_encoded: vector (nullable = true)



---

## Predict prepared data

In [13]:
from pyspark.sql import DataFrame
from src.utils import predict_with_model, get_data_dir

model_path = str(get_data_dir() / "model" / "gbt_cv_model_spark")

pred_result = predict_with_model(
    spark=spark,
    model_path=model_path,
    input_df=df_prepared,
    return_list=False,
)

pred_df: DataFrame = pred_result if isinstance(pred_result, DataFrame) else spark.createDataFrame(pred_result)
pred_df.show(10)

+---------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+
|timestamp_month|timestamp_year|resolution_time|     address_encoded|     latlong_encoded|organization_encoded|        type_encoded|            features|        prediction|
+---------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+
|              9|          2021|            275|(2048,[834,1804],...|[13.67891,100.66709]|(1786,[10,53],[1....|(25,[7,8],[1.0,1.0])|(3863,[0,1,836,18...|114.08362869937639|
|              9|          2021|            253|(2048,[348,426],[...| [13.7206,100.52649]|   (1786,[49],[1.0])|     (25,[14],[1.0])|(3863,[0,1,350,42...| 91.30143313782891|
|             12|          2021|            246|(2048,[802,1656],...| [13.8228,100.59165]|(1786,[31,108],[1...|(25,[0,8],[1.0,1.0])|(38

In [14]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="resolution_time", predictionCol="prediction", metricName="rmse"
)

rmse = evaluator.evaluate(pred_df)
print("RMSE:", rmse)

RMSE: 416.50532755420545


In [15]:
spark.stop()

---